# 08 Slow Job Debugging Playbook

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Debug a slow job end to end, the way you would at work or in an interview: measure a baseline, read the plan, check for skew and small files, list the problems, fix them one by one, verify that the results are unchanged, and measure again. You'll build a reusable <code>diagnose()</code> helper along the way.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
"A job that used to take 20 minutes now takes 3 hours. What do you do?" is the most common Databricks performance interview question, and a frequent real incident. A calm, repeatable process beats guessing at configuration settings.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 The playbook</b>
<ol><li><b>Baseline</b>: how long, how much data, what changed (data volume, code, cluster, upstream schema)?</li><li><b>Find the slow part</b>: the slowest job, stage and operator in the Spark UI or query profile</li><li><b>Read the plan</b> (<code>explain()</code> / profile): Python UDFs, the number of shuffles, join strategies, casts in join keys, filters that weren't pushed down</li><li><b>Check the data</b>: skewed keys (task time max vs median), row explosion, small files or huge files</li><li><b>Check resources</b>: spill, memory errors, retries, idle or too few tasks</li><li><b>Fix one thing at a time</b>, biggest first, and <b>verify the output is identical</b></li><li><b>Measure again</b> and write down what changed, for next time</li></ol>
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
After a marketing campaign, a revenue job went from 18 minutes to 2.5 hours. The playbook found it in 20 minutes: one task out of 200 ran for 2 hours (the profile showed skew), caused by a new "guest checkout" customer ID <code>'0'</code> that matched nothing anyway. Filtering placeholder IDs before the join, plus replacing a date-parsing UDF found in the same plan, brought the job to 11 minutes.
</div>

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Creates the data for a specimen job: 2 million orders and 50,000 customers. The data has realistic problems built in, which you'll discover through diagnosis rather than being told.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Row counts of both inputs.
</div>

In [0]:
DB = "workspace.optimization"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")

In [0]:
import time, io, re, contextlib
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.functions import udf

orders = spark.range(0, 2_000_000, 1, numPartitions=8).select(
    F.col("id").alias("order_id"),
    F.when(F.col("id") % 4 == 0, F.lit("0")).otherwise((F.col("id") % 50_000 + 1).cast("string")).alias("customer_id"),
    F.concat(F.lit("2024-"), F.lpad((F.col("id") % 12 + 1).cast("string"), 2, "0"), F.lit("-15")).alias("order_date_str"),
    (F.col("id") % 1000 / 10).alias("amount"),
)
customers = spark.range(1, 50_001).select(
    F.col("id").alias("customer_id"),
    F.element_at(F.array(F.lit("consumer"), F.lit("smb"), F.lit("enterprise")), (F.col("id") % 3 + 1).cast("int")).alias("segment"),
    F.sha2(F.col("id").cast("string"), 512).alias("notes"),
)
print("orders:", orders.count(), "| customers:", customers.count())

## The specimen: a slow job

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Defines the job as a team member wrote it: October revenue per customer segment, plus a detail table of October orders.<br><br>
<b>Why it matters</b><br>Every line looks reasonable on its own, which is why problems like these reach production. Don't fix anything yet: just run it and record the baseline.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The baseline time, the revenue per segment, and the number of files written for the detail table.
</div>

In [0]:
@udf("date")
def parse_date(s):
    return datetime.strptime(s, "%Y-%m-%d").date()

def slow_job():
    df = orders.withColumn("order_date", parse_date("order_date_str"))
    df = df.repartition(64)
    joined = df.join(customers, df.customer_id == customers.customer_id).drop(customers.customer_id)
    joined = joined.orderBy("order_id")
    october = joined.filter(F.month("order_date") == 10)
    summary = october.groupBy("segment").agg(F.round(F.sum("amount"), 2).alias("revenue"))
    return summary, october.repartition(64)

def run(job, table):
    start = time.time()
    summary, detail = job()
    rows = sorted(summary.collect())
    detail.write.mode("overwrite").saveAsTable(table)
    files = spark.sql(f"DESCRIBE DETAIL {table}").first()["numFiles"]
    return time.time() - start, rows, files

slow_time, slow_rows, slow_files = run(slow_job, f"{DB}.october_detail_slow")
print(f"baseline: {slow_time:.2f}s | detail files written: {slow_files}")
for r in slow_rows:
    print(r)

## Step 1: build a diagnosis helper

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes <code>diagnose(df)</code>, which reads the physical plan and reports the usual suspects: Python UDFs, the number of shuffles, sorts, join strategies, casts in join conditions, and where filters sit relative to Python evaluation.<br><br>
<b>Why it matters</b><br>Reading plans takes practice. A checklist in code makes the review fast and repeatable, and you can reuse this helper on your own jobs.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
A report for the slow job's summary query listing several problems.
</div>

In [0]:
def plan_text(df):
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        df.explain()
    return buf.getvalue()

def diagnose(df):
    p = plan_text(df)
    joins = [l.strip() for l in p.splitlines() if re.search(r"(SortMergeJoin|BroadcastHashJoin|ShuffledHashJoin)", l)]
    checks = {
        "Python UDF in plan (BatchEvalPython / ArrowEvalPython)": bool(re.search(r"(BatchEvalPython|ArrowEvalPython)", p)),
        "shuffles (Exchange, not broadcast)": len(re.findall(r"(?<!Broadcast)Exchange (hash|range|RoundRobin|SinglePartition)", p)),
        "global sorts (range partitioning)": len(re.findall(r"Exchange rangepartitioning", p)),
        "round-robin repartition": len(re.findall(r"RoundRobinPartitioning", p)),
        "join strategies": sorted({re.search(r"(SortMergeJoin|BroadcastHashJoin|ShuffledHashJoin)", j).group(1) for j in joins}),
        "cast in a join condition": any("cast(" in j for j in joins),
    }
    for k, v in checks.items():
        print(f"  {k:<55} {v}")
    return checks

summary_slow, _ = slow_job()
print("slow job:")
_ = diagnose(summary_slow)

## Step 2: check the data for skew

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Counts orders per join key and shows the heaviest keys and their share of all rows.<br><br>
<b>Why it matters</b><br>The plan can't show skew: you have to look at the data (or at task time distribution in the profile). A single key with a large share of rows means one task does most of the work in a shuffle join.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
Customer <code>'0'</code> holds 25% of all orders, far more than any real customer. It's also a placeholder ID that matches no customer.
</div>

In [0]:
total = orders.count()
(orders.groupBy("customer_id").count()
 .withColumn("share_pct", F.round(F.col("count") * 100 / total, 2))
 .orderBy(F.desc("count")).show(5))
print("customer '0' exists in customers:", customers.filter("customer_id = 0").count() > 0)

## Step 3: list the problems

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Summarizes the findings before fixing anything.<br><br>
<b>Why it matters</b><br>Writing the list down keeps the fixes focused, and it's exactly what an interviewer wants to hear: problem → evidence → fix.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
No output (a checklist).
</div>

| # | Problem | Evidence | Fix |
|---|---|---|---|
| 1 | Date parsed with a Python UDF | `BatchEvalPython` in the plan | `F.to_date` |
| 2 | Filter applied after the join, on a UDF result | Filter sits above the join in the plan | Filter first, on a built-in expression |
| 3 | Pointless `repartition(64)` and `orderBy` | Extra round-robin and range exchanges | Remove both |
| 4 | Join key type mismatch (string vs bigint) | `cast(` in the join condition | Cast once, before the join |
| 5 | 25% of rows on placeholder key `'0'` | Key distribution | Filter it out (it matches nothing) |
| 6 | Wide `notes` column carried through the join | The join output includes `notes`. Here the optimizer still broadcast `customers` because it's small, but on a larger dimension this becomes a `SortMergeJoin` that shuffles wide rows | Select needed columns, broadcast explicitly |
| 7 | 64 small files written for a small table | `numFiles` | Don't force 64 partitions on output |

## Step 4: fix, verify, measure

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Rewrites the job with all fixes, checks that the summary is identical, diagnoses the new plan, and compares time and output files.<br><br>
<b>Why it matters</b><br>A fix that changes the result isn't a fix. Always compare outputs before celebrating the speed-up.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The same revenue per segment, a clean diagnosis (no Python UDF, no casts, a broadcast join, 1 shuffle instead of 3), 8 output files instead of 64, and a run several times faster (about 9x locally).
</div>

In [0]:
def fast_job():
    october = (orders
               .filter((F.col("customer_id") != "0") & F.col("order_date_str").startswith("2024-10"))   # fixes 2, 5
               .withColumn("order_date", F.to_date("order_date_str"))                                  # fix 1
               .withColumn("customer_id", F.col("customer_id").cast("bigint"))                         # fix 4
               .join(F.broadcast(customers.select("customer_id", "segment")), "customer_id"))           # fix 6
    summary = october.groupBy("segment").agg(F.round(F.sum("amount"), 2).alias("revenue"))
    return summary, october                                                                             # fixes 3, 7

fast_time, fast_rows, fast_files = run(fast_job, f"{DB}.october_detail_fast")
print("same summary:", fast_rows == slow_rows)
print("same detail rows:", spark.table(f"{DB}.october_detail_fast").count() == spark.table(f"{DB}.october_detail_slow").count())
print("fast job:")
_ = diagnose(fast_job()[0])
print(f"\nslow: {slow_time:.2f}s, {slow_files} files | fast: {fast_time:.2f}s, {fast_files} files | {slow_time / fast_time:.1f}x faster")

## Step 5: record what you learned

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes a short incident note, the last step of the playbook.<br><br>
<b>Why it matters</b><br>Notes like this turn one fix into team knowledge: the next person who sees "BatchEvalPython" or a hot key <code>'0'</code> knows what to do.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The note, with your measured numbers filled in.
</div>

In [0]:
print(f"""INCIDENT NOTE: october revenue job
Symptom:   slow run ({slow_time:.1f}s at demo scale), {slow_files} output files
Causes:    Python date UDF, filter after join, redundant repartition + orderBy,
           string/bigint join key, 25% placeholder customer '0', wide dimension not broadcast
Fix:       built-ins, filter early, removed shuffles, typed key, filtered placeholder, broadcast narrow dimension
Result:    {fast_time:.1f}s, {fast_files} files, identical output
Follow-up: add assert_unique + key-type checks to Silver, alert on placeholder key share""")
for t in ["october_detail_slow", "october_detail_fast"]:
    spark.sql(f"DROP TABLE IF EXISTS {DB}.{t}")

## Under the hood

```
slow:  Scan ─ Python(parse_date) ─ RoundRobin(64) ─┐
       Scan customers (wide) ─────────────────────┴─ Join(cast key) ─ RangeSort ─ Filter(month=10) ─ Agg
       (broadcast here only because customers is small; a larger dimension means a SortMergeJoin of wide rows)
       25% of rows on key '0' ─▶ one hot task

fast:  Scan ─ Filter(not '0', starts with 2024-10) ─ to_date ─ cast ─┐
       Scan customers (2 cols) ─ BROADCAST ─────────────────────────┴─ BroadcastHashJoin ─ Agg
```

The fast plan reads 1/12 of the dates through the join, never leaves the JVM, and shuffles only the tiny final aggregate.

In [0]:
fast_job()[0].explain()

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: changing many configurations at once</b><br>
You won't know what helped, or what hurt. Fix one thing at a time and measure.<br><br>
<b>⛔ Problem: optimizing without checking the result</b><br>Always compare outputs (counts, sums or a full comparison on a sample) after each change.<br><br>
<b>⛔ Problem: blaming the cluster first</b><br>Most slow jobs are slow because of code and data (UDFs, skew, shuffles, small files). More nodes don't fix a single hot task.<br><br>
<b>⛔ Problem: "it was fast yesterday"</b><br>Look for what changed: data volume, a new hot key, a schema change (types), small files from a new upstream writer, a code change.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. A job that took 20 minutes now takes 3 hours. How do you debug it?</b><br>
First I find out what changed: data volume, code, cluster or upstream data. Then I open the Spark UI or query profile to find the slowest stage and operator, and check for skew (max task time vs median), spill, retries and the number of tasks. I read the plan for Python UDFs, extra shuffles, wrong join strategies, casts on join keys and filters that weren't pushed down, and I check the input for small files. I fix the biggest issue first, verify the output is identical, measure again, and document it.<br><br>

<b>🎤 2. How do you confirm that a performance fix didn't change the results?</b><br>
Compare outputs from the old and new versions: row counts, key aggregates, and a full <code>exceptAll</code> comparison on a sample or on the whole output when it's affordable.<br><br>

<b>🎤 3. What signs of trouble do you look for in a physical plan?</b><br>
<code>BatchEvalPython</code>/<code>ArrowEvalPython</code> (Python UDFs), many <code>Exchange</code> nodes (shuffles), <code>RoundRobinPartitioning</code> and <code>rangepartitioning</code> from unnecessary <code>repartition</code> and <code>orderBy</code>, <code>SortMergeJoin</code> where a broadcast would do, <code>cast</code> in join conditions, and filters that sit above joins or UDFs instead of near the scan.<br><br>

<b>🎤 4. Why does adding nodes often not help a skewed job?</b><br>
The stage waits for its slowest task, and a hot key's rows all go to one task. More nodes only make the other tasks finish earlier. Fixing the skew (AQE skew join, filtering placeholder keys, salting, broadcasting) is what shortens the stage.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. In a query profile, one task of a join stage runs for 40 minutes while the median task runs for 20 seconds. What is the most likely cause?</b><br>
A. Too many small files in the dimension table<br>
B. Data skew: a few join keys hold a large share of the rows<br>
C. The SQL warehouse is too large<br>
D. Photon is disabled<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. A physical plan contains <code>BatchEvalPython</code>. What does this indicate?</b><br>
A. A broadcast join<br>
B. A Python UDF is evaluated row by row outside the JVM<br>
C. The table uses deletion vectors<br>
D. Adaptive query execution is disabled<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Write your own slow version of a "revenue per month per segment" job with at least three of the problems from Step 3</li><li>Run <code>diagnose()</code> on it and list each problem with its evidence</li><li>Write the fast version, verify the results are identical, and report the speed-up</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. Slow version: UDF for month, wide join without broadcast, redundant repartition and orderBy
@udf("int")
def month_udf(s):
    return int(s[5:7])

slow = (orders.withColumn("month", month_udf("order_date_str")).repartition(32)
        .join(customers, orders.customer_id == customers.customer_id).orderBy("order_id")
        .groupBy("month", "segment").agg(F.round(F.sum("amount"), 2).alias("revenue")))

# 2. Diagnose
print("slow:")
_ = diagnose(slow)

# 3. Fast version: built-in substring, filter placeholder, typed key, narrow broadcast, no extra shuffles
fast = (orders.filter(F.col("customer_id") != "0")
        .withColumn("month", F.substring("order_date_str", 6, 2).cast("int"))
        .withColumn("customer_id", F.col("customer_id").cast("bigint"))
        .join(F.broadcast(customers.select("customer_id", "segment")), "customer_id")
        .groupBy("month", "segment").agg(F.round(F.sum("amount"), 2).alias("revenue")))
print("fast:")
_ = diagnose(fast)

t0 = time.time(); slow_result = sorted(slow.collect()); t1 = time.time(); fast_result = sorted(fast.collect()); t2 = time.time()
print(f"identical: {slow_result == fast_result} | slow {t1 - t0:.2f}s | fast {t2 - t1:.2f}s")

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Baseline → find the slow stage → read the plan → check the data → fix one thing → verify → measure → document</li><li>Plan red flags: Python UDFs, extra exchanges, sort-merge where broadcast fits, casts on join keys, late filters</li><li>Data red flags: hot or placeholder keys, duplicate keys, small files</li><li>Never accept a speed-up without proving the results are identical</li><li>Code and data fixes usually beat bigger clusters</li></ul>
</div>

| Red flag | Where | Fix |
|---|---|---|
| `BatchEvalPython` | Plan | Built-in functions |
| Many `Exchange` | Plan | Remove repartition, orderBy, join-backs |
| `cast(` in join | Plan | Align key types in Silver |
| One long task | Profile, key counts | Filter placeholders, AQE skew, salting |
| Many tiny files | `DESCRIBE DETAIL` | Stop forcing partitions, `OPTIMIZE` |

## Git commit

```
git add 05_optimization/08_slow_job_debugging_playbook.ipynb
git commit -m "add 05_08 slow job debugging playbook notebook"
```